In [1]:
# Load Packages
from datetime import date, datetime, timedelta
import calendar
import pandas as pd
import numpy as np
import itertools
import copy
from numpy.linalg import multi_dot
from scipy.stats import norm
from scipy.stats import bernoulli
import matplotlib.pyplot as plt
import math



#Import Date opertative user defined functions
from ipynb.fs.full.user_defined_vik_functions import get_all_monthly_option_expiries, \
                                                     find_last_thurs_date_of_month, \
                                                     prev_workday_if_holiday, find_wkly_expries,\
                                                     date_of_prev_thurs, generate_daily_dates_each_month, \
                                                     next_workday, next_weekday, \
                                                     create_df_wk_days, find_week, date_of_next_thurs

# Import dataframe naming functions 
from ipynb.fs.full.user_defined_vik_functions import get_mthly_df_name_from_expiry

#Import data loading functions
from ipynb.fs.full.user_defined_vik_functions import load_all_mthly_data



In [2]:
#Compute Daily Static  Hedging Errors

###################################
# Common Variables initialisation #
###################################

# Stock Index of Interest
stock_ident = "BANKNIFTY"

#Path to refer data
source_path = "/home/jupyter-partha/Vikranth - Chapter 2/"
input_sub_path = "Input Data/mkt_data_covid_region/"
output_sub_path = "Output Data/"
input_data_path = source_path + input_sub_path
output_data_path = source_path + output_sub_path

#Static hedging performed at different moneyness regions 
#i.e. moneyness is used to select the option with nearest moneyness match
# ATM - At the Money, ITM - In the money, OTM - Out of the Money
prod_moneyness = "ATM"

#Product type to hedge: either "CE" or "PE"
prod_type = "CE"

# Periods of interest will be a dictionary
#Key is the year, value is a list of months 1-12, 1- Jan, 2 - Feb,...12 - Dec
# For E.g., periods_of_interest = {2020: [3], 2019: [11, 12]}
periods_of_interest = {2020: [3]}

#List of holidays
holidays_list = [date(2020, 3, 10)]


In [3]:

# The function pulls the option value of the trade to be hedge for each day
def generate_dict_opt_value(dict_daily_dates_error_cal, monthly_mkt_data, holidays_list, \
                            prod_type, output_path, stock_ident):
    dict_monthly_option_value = {}
    full_dates_list = []
    full_price_list = []
    df_strikes_hedged = pd.read_csv( output_path + "B4_" + stock_ident + "_" + prod_type  + "_strikes_hedged.csv")
    for each_month in dict_daily_dates_error_cal.keys():
        each_month_data = monthly_mkt_data[get_mthly_df_name_from_expiry(datetime.strptime(each_month, "%d-%b-%Y").date(), prod_type, holidays_list, stock_ident)]
        strike = df_strikes_hedged[df_strikes_hedged["Date"] == each_month]["Strike"].iloc[0]
        df_opt_values = each_month_data[each_month_data["Strike Price"] == strike]
        df_opt_values.assign(x=pd.to_datetime(df_opt_values['Date'])).sort_values('x').drop('x', 1)
        dates_list = df_opt_values["Date"].tolist()[1:]
        full_dates_list = full_dates_list + dates_list
        price_list = df_opt_values["Close"].tolist()[1:]
        full_price_list = full_price_list + price_list
    
    dict_monthly_option_value = {"Date": full_dates_list , "Close Price": full_price_list}
    df_monthly_option_value = pd.DataFrame(dict_monthly_option_value)
    
    return df_monthly_option_value

def combine_weekly_option_df(dict_wkly_expiries_each_month, prod_type, input_path, stock_ident):
    list_of_weekly_df = []
    for each_month in dict_wkly_expiries_each_month.keys():
        for week in range(0,4):
            if (week < 3):
                start_date = dict_wkly_expiries_each_month[each_month][week].strftime("%d-%b-%Y") 
                expiry_date = dict_wkly_expiries_each_month[each_month][week+1].strftime("%d-%b-%Y")
            else:
                start_date = dict_wkly_expiries_each_month[each_month][week].strftime("%d-%b-%Y") 
                expiry_date = each_month
            
            file_name = "OPTIDX_" + stock_ident + "_" + prod_type + "_" \
                        + start_date +  "_TO_" \
                        + expiry_date + ".csv" 
            df = pd.read_csv(input_path + file_name)
            df = df[df['Date'] != dict_wkly_expiries_each_month[each_month][week].strftime("%d-%b-%Y") ]
            
            list_of_weekly_df.append(df)
    
    weekly_df_prod = pd.concat(list_of_weekly_df, axis=0) 
    return weekly_df_prod


In [4]:
def find_daily_static_hedging_error(df_wk_day_map, df_ce_weekly, df_pe_weekly,\
                                    df_monthly_option_value,\
                                    prod_type, prod_moneyness, \
                                    holidays_list, output_path):

    df_ce_map = pd.merge(df_wk_day_map, df_ce_weekly, how = 'left', left_on = ['Day'], right_on = ['Date'])
    df_pe_map = pd.merge(df_wk_day_map, df_pe_weekly, how = 'left', left_on = ['Day'], right_on = ['Date'])
    df_ce_map = df_ce_map[['Week', 'Day', 'Strike Price', 'Close']]
    df_pe_map = df_pe_map[['Week', 'Day', 'Strike Price', 'Close']]
#     df_ce_map.to_csv(output_data_path + "C1_" + prod_type +  "_df_ce_map.csv")
    
    ce_weights = pd.read_csv(output_path + "B4_" + stock_ident + "_" + "CE_Pfl_weights_const_Vol_Surf.csv")
    pe_weights = pd.read_csv(output_path + "B4_" + stock_ident + "_" + "PE_Pfl_weights_const_Vol_Surf.csv")
    cash_invst = pd.read_csv(output_path + "B4_" + stock_ident + "_" + "Cash_Pfl_weights_const_Vol_Surf.csv")
    
    df_ce_pfl = pd.merge(df_ce_map, ce_weights, how = 'left', \
                         left_on = ['Week', 'Strike Price'], \
                         right_on = ['Date', 'Strike'])
    
    df_ce_pfl['Pfl_weights'].replace('', np.nan, inplace=True)
    df_ce_pfl.dropna(subset=['Pfl_weights'], inplace=True)
    
    df_ce_pfl = df_ce_pfl[['Week', 'Day', 'Strike Price', 'Close', 'Pfl_weights']]
    df_ce_pfl['hedge_value'] = df_ce_pfl['Close'] * df_ce_pfl['Pfl_weights'] 
#     df_ce_pfl.to_csv(output_data_path + "C1_" + prod_type +  "_df_ce_pfl.csv")
    df_ce_pfl = df_ce_pfl.groupby('Day')[['hedge_value']].sum().reset_index()
    df_ce_pfl = df_ce_pfl.assign(x=pd.to_datetime(df_ce_pfl['Day'])).sort_values('x').drop('x', 1).reset_index(drop=True)
#     df_ce_pfl.to_csv(output_data_path + "C2_" + prod_type +  "_df_ce_pfl.csv")
    
    
    df_pe_pfl = pd.merge(df_pe_map, pe_weights, how = 'left', \
                         left_on = ['Week', 'Strike Price'], \
                         right_on = ['Date', 'Strike'])
    df_pe_pfl['Pfl_weights'].replace('', np.nan, inplace=True)
    df_pe_pfl.dropna(subset=['Pfl_weights'], inplace=True)
    
    df_pe_pfl = df_pe_pfl[['Week', 'Day', 'Strike Price', 'Close', 'Pfl_weights']]
    df_pe_pfl['hedge_value'] = df_pe_pfl['Close'] * df_pe_pfl['Pfl_weights']  
    df_pe_pfl = df_pe_pfl.groupby('Day')[['hedge_value']].sum().reset_index()
    df_pe_pfl = df_pe_pfl.assign(x=pd.to_datetime(df_pe_pfl['Day'])).sort_values('x').drop('x', 1).reset_index(drop=True)
#     df_pe_pfl.to_csv(output_data_path + "C2_" + prod_type +  "_df_Pe_pfl.csv")
    
    #########################################
    #Constant Interest rate to be addressed 
    ########################################
    r = 0.05
    df_cash_pfl = pd.merge(df_wk_day_map, cash_invst, how = 'left', \
                         left_on = ['Week'], \
                         right_on = ['Date'])
    df_cash_pfl = df_cash_pfl[['Week', 'Day', 'Pfl_weights']]
    df_cash_pfl['Week_end'] = pd.to_datetime(df_cash_pfl['Day'], format= "%d-%b-%Y").apply(lambda x: prev_workday_if_holiday(date_of_next_thurs(x), holidays_list))
    df_cash_pfl['time_value'] = (pd.to_datetime(df_cash_pfl['Week_end'], format= "%d-%b-%Y") - pd.to_datetime(df_cash_pfl['Day'], format= "%d-%b-%Y")).dt.days / 365
    df_cash_pfl['Factor'] = df_cash_pfl['time_value'].apply(lambda x: np.exp(- x * r))
    df_cash_pfl['hedge_value'] = df_cash_pfl['Factor'] * df_cash_pfl['Pfl_weights']
#     df_cash_pfl.to_csv(output_data_path + "C2_" + "cash.csv")
    
    df_static_hedge = df_ce_pfl.merge(df_pe_pfl , how = 'left', left_on='Day', right_on='Day',\
                                suffixes=('_CE', '_PE'))
    df_static_hedge = df_static_hedge.merge(df_cash_pfl,  how = 'left', left_on='Day', right_on='Day')
    df_static_hedge['Hedge_Pfl_value'] = df_static_hedge['hedge_value_CE'] + \
                                         df_static_hedge['hedge_value_PE'] + \
                                         df_static_hedge['hedge_value']
    
    df_static_hedge = df_static_hedge[['Day', 'Hedge_Pfl_value']]
    df_static_hedge = pd.merge(df_static_hedge, df_monthly_option_value, how = 'left', left_on = ['Day'], right_on = ['Date'])
    df_static_hedge['Static Hedge Error (Const Vol Surf)'] = df_static_hedge['Close Price'] - df_static_hedge['Hedge_Pfl_value']
    
    df_static_hedge = df_static_hedge[['Date', 'Close Price', 'Hedge_Pfl_value', 'Static Hedge Error (Const Vol Surf)']]
    df_static_hedge.to_csv(output_path + "C4_" + stock_ident + "_" + prod_type + "_BS_const_Vol_Surf_static_hedge_error.csv", index = False)
    
    return df_static_hedge



In [5]:

#Find the monthly strike of option from mkt data to find the option to be hedged
#Every month, an option is hedged
mthly_expiries_list = get_all_monthly_option_expiries(periods_of_interest, holidays_list)
dict_wkly_expiries_each_month = find_wkly_expries(mthly_expiries_list, holidays_list)
dict_daily_dates_error_cal = generate_daily_dates_each_month(dict_wkly_expiries_each_month, holidays_list)

# #Load all monthly mkt data
mthly_mkt_data = load_all_mthly_data(mthly_expiries_list, input_data_path, holidays_list, prod_type_lists=["FUT", "CE", "PE"], stock_ident = "BANKNIFTY")

df_wk_day_map = create_df_wk_days(dict_daily_dates_error_cal, dict_wkly_expiries_each_month, holidays_list)


df_monthly_option_value = generate_dict_opt_value(dict_daily_dates_error_cal,mthly_mkt_data, holidays_list, \
                          prod_type=prod_type, output_path=output_data_path, stock_ident=stock_ident)


df_ce_weekly = combine_weekly_option_df(dict_wkly_expiries_each_month, prod_type="CE", 
                                        input_path=input_data_path, stock_ident=stock_ident)
df_pe_weekly = combine_weekly_option_df(dict_wkly_expiries_each_month, prod_type="PE", 
                                        input_path=input_data_path, stock_ident=stock_ident)

df_static_hedge = find_daily_static_hedging_error(df_wk_day_map, df_ce_weekly, df_pe_weekly, df_monthly_option_value, \
                                           prod_type=prod_type, prod_moneyness=prod_moneyness, holidays_list=holidays_list, output_path=output_data_path)


#Final output
#DataFrame: "Date", "Option Value", "Quantity" - this to be considered later, "Static Hedging Error", "Dynamic Hedging Error"
